In [ ]:
import os, sys

print(os.path.abspath(os.path.join(os.getcwd(), '.')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from extern.network_backend.g2.network_v5 import Network

In [ ]:
import sys
from collections import deque
import numpy as np

def get_deep_size(obj, seen=None):
    """
    Recursively estimate the memory footprint of an object in bytes.
    Works with builtins, numpy arrays, and falls back safely for unknown objects.
    """
    if seen is None:
        seen = set()
        
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)

    size = sys.getsizeof(obj)

    # Special case: numpy arrays
    if isinstance(obj, np.ndarray):
        size += obj.nbytes

    # Built-in containers
    elif isinstance(obj, dict):
        size += sum(get_deep_size(k, seen) + get_deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset, deque)):
        size += sum(get_deep_size(i, seen) for i in obj)

    # Objects with __dict__ (normal Python classes)
    elif hasattr(obj, "__dict__"):
        size += get_deep_size(vars(obj), seen)

    # Objects with __slots__ (Python classes with slots)
    elif hasattr(obj, "__slots__"):
        for slot in obj.__slots__:
            if hasattr(obj, slot):
                value = getattr(obj, slot)
                if value is not None:
                    size += get_deep_size(value, seen)

    # Else: fallback → just the shallow size
    return size



In [ ]:
import g2_lib as g2

import pandas as pd
import random
from itertools import product
import numpy as np
import copy
import psutil

# Define network configurations to test
npus_counts = [[32], [64], [128], [1024], [8, 8], [4, 4, 4]]
topologies = [['ring'], ['switch'], ['ring', 'switch'], ['ring', 'ring']]

# Define the number of routes to add in each iteration
route_counts = [5, 20, 50, 100, 500]

results = []

process = psutil.Process(os.getpid())
# Loop through each configuration
for npus_count in npus_counts:
    # Ensure topology dimension matches npus_count dimension
    for topo in topologies:
        if len(npus_count) != len(topo):
            continue

        num_npus = np.prod(npus_count)
        config_name = f"npus:{npus_count}-topo:{topo}"
        print(f"--- Testing Configuration: {config_name} ---")

        # Loop through the number of routes to add
        for num_routes in route_counts:
            
            # Measure memory before creating the network object
            mem_before_net = process.memory_info().rss
            
            # 1. Initialize the Network
            net = Network(
                npus_count_per_dim=npus_count,
                bandwidth_per_dim=[900] * len(npus_count),
                topologies_per_dim=topo,
            )

            # 2. Add routes
            for i in range(num_routes):
                src = random.randint(0, num_npus - 1)
                dest = random.randint(0, num_npus - 1)
                # Ensure src and dest are different
                while src == dest:
                    dest = random.randint(0, num_npus - 1)
                
                # Add a sample route
                net.add_route(
                    tag=i,
                    src=src,
                    size=90,
                    dest=dest,
                    chunk_id=i,
                    workload_node_id=f"workload_{i}"
                )

            # 3. Build the network object from the defined links and routes
            net.get_next_messages(2)
            #net.network = g2.network_from_json_objs(net.network_links_json, net.routes_to_add_json)
            
            # Measure memory after creating the network object
            mem_after_net = process.memory_info().rss
            psutil_net_kb = (mem_after_net - mem_before_net) / 1024

            # Create a dictionary and fill it with copies of the network
            num_copies = 500  # Number of copies to store
            network_dict = {}
            key_name = ''.join(net.network.flow_names())
            for i in range(num_copies):
                network_dict[f"{key_name}_{i}"] = net.network

            # Measure memory after creating the dictionary
            mem_after_dict = process.memory_info().rss
            psutil_dict_kb = (mem_after_dict - mem_after_net) / 1024

            # 4. Calculate the deep size of the network object
            size_in_bytes = get_deep_size(net.network)
            size_in_kb = size_in_bytes / 1024

            size_in_bytes_dict = get_deep_size(network_dict)
            size_in_kb_dict = size_in_bytes_dict / 1024
            
            print(f"  Routes: {num_routes}, Size: {size_in_kb:.2f} KB, Size dict: {size_in_kb_dict:.2f} KB")
            print(f"  psutil - Net mem: {psutil_net_kb:.2f} KB, Dict mem: {psutil_dict_kb:.2f} KB")
            # Store results
            results.append({
                'config': config_name,
                'npus_count': str(npus_count),
                'topology': str(topo),
                'num_routes': num_routes,
                'size_kb': size_in_kb,
                'size_kb_dict': size_in_kb_dict
            })
            
            # Clean up for the next iteration
            del net

# Create a DataFrame from the results for easier analysis
df_results = pd.DataFrame(results)
print("\n--- Comparison Summary ---")
print(df_results)

# You can now easily plot the results, for example:
# df_results.pivot(index='num_routes', columns='config', values='size_kb').plot()

In [ ]:
import 